---
## 7. Config A — Hyperparameter Optimisation

The model-free agents (SARSA and Q-Learning) trained in Sections 3 and 4 used default hyperparameters and only 50,000 episodes. Looking at the learning curves, both agents were still improving at the end of training — meaning they had not fully converged yet.

In this section we systematically investigate two questions:

1. **Does more training help?** We run Q-Learning with 50k, 200k, and 500k episodes and compare survival rates.
2. **What is the best learning rate alpha?** We run a grid search over alpha ∈ {0.05, 0.1, 0.2, 0.3, 0.5} and compare final performance.

The goal is to get the model-free agents as close to the Policy Iteration ceiling (78.8%) as possible, while understanding what drives the remaining gap.


### 7.1 — Effect of Number of Training Episodes

If an agent is still learning at 50,000 episodes, running longer should improve its policy. We test Q-Learning (which performed best in Section 4) with three episode counts: 50k (baseline already done), 200k, and 500k.

We keep alpha=0.3 fixed here so we are only changing one thing at a time.


In [ ]:
# ── Effect of training length on Q-Learning ───────────────────────────────────
# We already have ql_results from 50k episodes (Section 4)
# Now run 200k and 500k, keeping alpha=0.3 fixed

episode_counts = [200_000, 500_000]
episode_results = {
    50_000: ql_results   # already computed in Section 4
}
episode_returns = {
    50_000: ql_returns   # already computed in Section 4
}

for n_ep in episode_counts:
    print(f'\nTraining Q-Learning with {n_ep:,} episodes...')
    Q_tmp, returns_tmp = q_learning(
        n_episodes=n_ep,
        alpha=0.3,
        gamma=GAMMA,
        epsilon_start=1.0,
        epsilon_min=0.01,
        seed=SEED,
    )
    policy_tmp = np.argmax(Q_tmp, axis=1)
    results_tmp = evaluate_policy_tabular(policy_tmp)
    episode_results[n_ep] = results_tmp
    episode_returns[n_ep] = returns_tmp
    print(f'  Survival rate : {results_tmp["survival_rate"]:.1f}%')
    print(f'  Mean return   : {results_tmp["mean_return"]:.4f}')

print('\nDone.')


In [ ]:
# ── Plot: survival rate vs number of training episodes ────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: bar chart of survival rate per episode count
ep_labels   = [f'{n//1000}k' for n in sorted(episode_results.keys())]
ep_survival = [episode_results[n]['survival_rate'] for n in sorted(episode_results.keys())]

bars = axes[0].bar(ep_labels, ep_survival, color='darkorange', alpha=0.85, edgecolor='white')
axes[0].axhline(pi_results['survival_rate'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[0].axhline(survival_rate, color='gray', linestyle=':',
                linewidth=1.5, label=f'Random baseline ({survival_rate:.1f}%)')
axes[0].set_xlabel('Training episodes')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Q-Learning: Survival Rate vs Training Length\n(alpha=0.3 fixed)', fontweight='bold')
axes[0].set_ylim(60, 85)
axes[0].legend(fontsize=9)
for bar, val in zip(bars, ep_survival):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Right: learning curves overlaid for all three runs
window = 1000
colors_ep = ['#f5a623', '#e07b00', '#8b4500']
for (n_ep, color) in zip(sorted(episode_returns.keys()), colors_ep):
    rolling = pd.Series(episode_returns[n_ep]).rolling(window).mean()
    axes[1].plot(rolling, color=color, linewidth=1.8, label=f'{n_ep//1000}k episodes')

axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--',
                linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('Q-Learning: Learning Curves by Training Length\n(alpha=0.3 fixed)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_episode_count_comparison.png', bbox_inches='tight')
plt.show()

print('Interpretation:')
print('  - If the bars keep rising: the agent is still learning, more episodes help.')
print('  - If the bars plateau: the agent has converged; more episodes give no benefit.')
print('  - The gap to PI ceiling shows how much the lack of the model costs.')


### 7.2 — Alpha Grid Search (Learning Rate)

The learning rate alpha controls how strongly the agent updates its Q-values after each new experience.

- **Too small (e.g. 0.05)**: the agent updates very slowly and may not converge within our episode budget.
- **Too large (e.g. 0.5)**: the agent reacts too aggressively to single experiences, making Q-values unstable.
- **Just right**: fast enough to learn, stable enough to converge.

We test five values of alpha using Q-Learning with 200k episodes (a good balance of quality and compute time). Everything else is kept constant.


In [ ]:
# ── Alpha grid search ─────────────────────────────────────────────────────────
# Run Q-Learning at 200k episodes for each alpha value
# Everything else held constant: gamma, epsilon schedule, seed

alpha_values  = [0.05, 0.1, 0.2, 0.3, 0.5]
alpha_results = {}
alpha_returns = {}

for alpha_val in alpha_values:
    print(f'Training Q-Learning | alpha={alpha_val} | 200k episodes...')
    Q_tmp, returns_tmp = q_learning(
        n_episodes=200_000,
        alpha=alpha_val,
        gamma=GAMMA,
        epsilon_start=1.0,
        epsilon_min=0.01,
        seed=SEED,
    )
    policy_tmp = np.argmax(Q_tmp, axis=1)
    results_tmp = evaluate_policy_tabular(policy_tmp)
    alpha_results[alpha_val] = results_tmp
    alpha_returns[alpha_val] = returns_tmp
    print(f'  Survival rate : {results_tmp["survival_rate"]:.1f}%  |  '
          f'Mean return : {results_tmp["mean_return"]:.4f}')

print('\nGrid search complete.')

# Find best alpha
best_alpha = max(alpha_results, key=lambda a: alpha_results[a]['survival_rate'])
print(f'\nBest alpha: {best_alpha} '
      f'(survival rate: {alpha_results[best_alpha]["survival_rate"]:.1f}%)')


In [ ]:
# ── Plot: alpha grid search results ───────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: survival rate per alpha
alpha_labels   = [str(a) for a in alpha_values]
alpha_survival = [alpha_results[a]['survival_rate'] for a in alpha_values]

bar_colors = ['#f5cba7' if a != best_alpha else 'darkorange' for a in alpha_values]
bars = axes[0].bar(alpha_labels, alpha_survival, color=bar_colors, edgecolor='white', alpha=0.9)
axes[0].axhline(pi_results['survival_rate'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[0].axhline(survival_rate, color='gray', linestyle=':',
                linewidth=1.5, label=f'Random baseline ({survival_rate:.1f}%)')
axes[0].set_xlabel('Learning rate (alpha)')
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Q-Learning: Survival Rate vs Alpha\n(200k episodes, highlighted = best)',
                  fontweight='bold')
axes[0].set_ylim(60, 85)
axes[0].legend(fontsize=9)
for bar, val in zip(bars, alpha_survival):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Right: learning curves for all alpha values
cmap = plt.cm.YlOrRd
alpha_color_map = {a: cmap(0.3 + i * 0.14) for i, a in enumerate(alpha_values)}
window = 1000

for alpha_val in alpha_values:
    rolling = pd.Series(alpha_returns[alpha_val]).rolling(window).mean()
    lw = 2.5 if alpha_val == best_alpha else 1.2
    axes[1].plot(rolling, color=alpha_color_map[alpha_val],
                 linewidth=lw, label=f'alpha={alpha_val}'
                 + (' (best)' if alpha_val == best_alpha else ''))

axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--',
                linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('Q-Learning: Learning Curves by Alpha\n(200k episodes)', fontweight='bold')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_alpha_grid_search.png', bbox_inches='tight')
plt.show()

print('Interpretation:')
print(f'  - Best alpha found: {best_alpha}')
print('  - Small alpha (0.05): slow to converge, may still be climbing at 200k.')
print('  - Large alpha (0.5): fast initial learning but may oscillate/destabilise.')
print('  - Medium alpha: best balance of speed and stability.')


### 7.3 — Best Configuration: Final Optimised Run

Using the best alpha found in the grid search, we now run the **final optimised version** of both Q-Learning and SARSA with:
- Best alpha from grid search
- 500k training episodes
- Same epsilon decay schedule

We also apply the same optimisation to SARSA for a fair comparison.


In [ ]:
# ── Optimised Q-Learning — best alpha, 500k episodes ─────────────────────────
print(f'Training OPTIMISED Q-Learning | alpha={best_alpha} | 500k episodes...')

ql_opt_Q, ql_opt_returns = q_learning(
    n_episodes=500_000,
    alpha=best_alpha,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
)

ql_opt_policy  = np.argmax(ql_opt_Q, axis=1)
ql_opt_results = evaluate_policy_tabular(ql_opt_policy)

print(f'  Survival rate : {ql_opt_results["survival_rate"]:.1f}%')
print(f'  Mean return   : {ql_opt_results["mean_return"]:.4f}')
print(f'  vs original QL: {ql_results["survival_rate"]:.1f}%')
print(f'  vs PI ceiling : {pi_results["survival_rate"]:.1f}%')


In [ ]:
# ── Optimised SARSA — best alpha, 500k episodes ───────────────────────────────
print(f'Training OPTIMISED SARSA | alpha={best_alpha} | 500k episodes...')

sarsa_opt_Q, sarsa_opt_returns = sarsa(
    n_episodes=500_000,
    alpha=best_alpha,
    gamma=GAMMA,
    epsilon_start=1.0,
    epsilon_min=0.01,
    seed=SEED,
)

sarsa_opt_policy  = np.argmax(sarsa_opt_Q, axis=1)
sarsa_opt_results = evaluate_policy_tabular(sarsa_opt_policy)

print(f'  Survival rate : {sarsa_opt_results["survival_rate"]:.1f}%')
print(f'  Mean return   : {sarsa_opt_results["mean_return"]:.4f}')
print(f'  vs original SARSA: {sarsa_results["survival_rate"]:.1f}%')
print(f'  vs PI ceiling    : {pi_results["survival_rate"]:.1f}%')


### 7.4 — Original vs Optimised: Full Comparison

We now compare the original (default) and optimised versions of all model-free agents side by side. This answers the key question: **how much did hyperparameter tuning and longer training actually help?**


In [ ]:
# ── Full comparison table: original vs optimised ──────────────────────────────

comparison_table = pd.DataFrame([
    {
        'Algorithm'        : 'Random Baseline',
        'Version'          : '—',
        'Episodes'         : '—',
        'Alpha'            : '—',
        'Survival Rate %'  : round(survival_rate, 1),
        'Mean Return'      : round(float(np.mean(rand_returns)), 4),
        'Gap to PI ceiling': round(pi_results['survival_rate'] - survival_rate, 1),
    },
    {
        'Algorithm'        : 'Policy Iteration',
        'Version'          : 'Optimal (ceiling)',
        'Episodes'         : '—',
        'Alpha'            : '—',
        'Survival Rate %'  : round(pi_results['survival_rate'], 1),
        'Mean Return'      : round(pi_results['mean_return'], 4),
        'Gap to PI ceiling': 0.0,
    },
    {
        'Algorithm'        : 'SARSA',
        'Version'          : 'Original',
        'Episodes'         : '50k',
        'Alpha'            : '0.3',
        'Survival Rate %'  : round(sarsa_results['survival_rate'], 1),
        'Mean Return'      : round(sarsa_results['mean_return'], 4),
        'Gap to PI ceiling': round(pi_results['survival_rate'] - sarsa_results['survival_rate'], 1),
    },
    {
        'Algorithm'        : 'SARSA',
        'Version'          : 'Optimised',
        'Episodes'         : '500k',
        'Alpha'            : str(best_alpha),
        'Survival Rate %'  : round(sarsa_opt_results['survival_rate'], 1),
        'Mean Return'      : round(sarsa_opt_results['mean_return'], 4),
        'Gap to PI ceiling': round(pi_results['survival_rate'] - sarsa_opt_results['survival_rate'], 1),
    },
    {
        'Algorithm'        : 'Q-Learning',
        'Version'          : 'Original',
        'Episodes'         : '50k',
        'Alpha'            : '0.3',
        'Survival Rate %'  : round(ql_results['survival_rate'], 1),
        'Mean Return'      : round(ql_results['mean_return'], 4),
        'Gap to PI ceiling': round(pi_results['survival_rate'] - ql_results['survival_rate'], 1),
    },
    {
        'Algorithm'        : 'Q-Learning',
        'Version'          : 'Optimised',
        'Episodes'         : '500k',
        'Alpha'            : str(best_alpha),
        'Survival Rate %'  : round(ql_opt_results['survival_rate'], 1),
        'Mean Return'      : round(ql_opt_results['mean_return'], 4),
        'Gap to PI ceiling': round(pi_results['survival_rate'] - ql_opt_results['survival_rate'], 1),
    },
    {
        'Algorithm'        : 'Double Q-Learning',
        'Version'          : 'Original',
        'Episodes'         : '50k',
        'Alpha'            : '0.3',
        'Survival Rate %'  : round(dql_results['survival_rate'], 1),
        'Mean Return'      : round(dql_results['mean_return'], 4),
        'Gap to PI ceiling': round(pi_results['survival_rate'] - dql_results['survival_rate'], 1),
    },
])

display(comparison_table)
comparison_table.to_csv(f'{PLOTS_DIR}/configA_optimisation_comparison.csv', index=False)


In [ ]:
# ── Visual comparison: original vs optimised survival rates ───────────────────

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: grouped bar chart original vs optimised
algorithms   = ['SARSA', 'Q-Learning']
orig_survival = [sarsa_results['survival_rate'], ql_results['survival_rate']]
opt_survival  = [sarsa_opt_results['survival_rate'], ql_opt_results['survival_rate']]

x = np.arange(len(algorithms))
width = 0.35

bars_orig = axes[0].bar(x - width/2, orig_survival, width,
                        label='Original (50k, alpha=0.3)',
                        color='#aed6f1', edgecolor='white', alpha=0.9)
bars_opt  = axes[0].bar(x + width/2, opt_survival, width,
                        label=f'Optimised (500k, alpha={best_alpha})',
                        color='steelblue', edgecolor='white', alpha=0.9)

axes[0].axhline(pi_results['survival_rate'], color='tomato', linestyle='--',
                linewidth=1.5, label=f'PI ceiling ({pi_results["survival_rate"]:.1f}%)')
axes[0].axhline(survival_rate, color='gray', linestyle=':',
                linewidth=1.5, label=f'Random ({survival_rate:.1f}%)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(algorithms)
axes[0].set_ylabel('Survival Rate (%)')
axes[0].set_title('Original vs Optimised: Survival Rate', fontweight='bold')
axes[0].set_ylim(60, 85)
axes[0].legend(fontsize=8)

for bar, val in zip(list(bars_orig) + list(bars_opt),
                    orig_survival + opt_survival):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

# Right: optimised learning curves vs PI ceiling
window = 2000
sarsa_opt_roll = pd.Series(sarsa_opt_returns).rolling(window).mean()
ql_opt_roll    = pd.Series(ql_opt_returns).rolling(window).mean()

# Show original (faded) and optimised (solid) for each algorithm
sarsa_orig_roll = pd.Series(sarsa_returns).rolling(500).mean()
ql_orig_roll    = pd.Series(ql_returns).rolling(500).mean()

axes[1].plot(sarsa_orig_roll, color='steelblue', linewidth=1.0,
             alpha=0.35, label='SARSA original (50k)')
axes[1].plot(ql_orig_roll, color='darkorange', linewidth=1.0,
             alpha=0.35, label='Q-Learning original (50k)')
axes[1].plot(sarsa_opt_roll, color='steelblue', linewidth=2.2,
             label=f'SARSA optimised (500k, alpha={best_alpha})')
axes[1].plot(ql_opt_roll, color='darkorange', linewidth=2.2,
             label=f'Q-Learning optimised (500k, alpha={best_alpha})')
axes[1].axhline(pi_results['mean_return'], color='tomato', linestyle='--',
                linewidth=1.5, label='PI ceiling')
axes[1].axhline(np.mean(rand_returns), color='gray', linestyle=':',
                linewidth=1.5, label='Random baseline')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Rolling mean return')
axes[1].set_title('Original vs Optimised: Learning Curves\n(faded = original, solid = optimised)',
                  fontweight='bold')
axes[1].legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{PLOTS_DIR}/configA_optimised_comparison.png', bbox_inches='tight')
plt.show()


In [ ]:
# ── Final optimisation summary ────────────────────────────────────────────────

print('=' * 65)
print('CONFIG A — OPTIMISATION SUMMARY')
print('=' * 65)
print(f'{"Algorithm":<25} {"Version":<15} {"Survival %":>12} {"Gap to PI":>10}')
print('-' * 65)

rows = [
    ('Random Baseline',   '—',          survival_rate,                   pi_results['survival_rate'] - survival_rate),
    ('Policy Iteration',  'Ceiling',     pi_results['survival_rate'],     0.0),
    ('SARSA',             'Original',    sarsa_results['survival_rate'],  pi_results['survival_rate'] - sarsa_results['survival_rate']),
    ('SARSA',             'Optimised',   sarsa_opt_results['survival_rate'], pi_results['survival_rate'] - sarsa_opt_results['survival_rate']),
    ('Q-Learning',        'Original',    ql_results['survival_rate'],     pi_results['survival_rate'] - ql_results['survival_rate']),
    ('Q-Learning',        'Optimised',   ql_opt_results['survival_rate'], pi_results['survival_rate'] - ql_opt_results['survival_rate']),
]

for name, version, sr, gap in rows:
    print(f'{name:<25} {version:<15} {sr:>11.1f}% {gap:>9.1f}pp')

print('=' * 65)
print()

# Improvement summary
sarsa_improvement = sarsa_opt_results['survival_rate'] - sarsa_results['survival_rate']
ql_improvement    = ql_opt_results['survival_rate']    - ql_results['survival_rate']

print('Improvement from optimisation:')
print(f'  SARSA:      +{sarsa_improvement:.1f} percentage points')
print(f'  Q-Learning: +{ql_improvement:.1f} percentage points')
print()
print('Remaining gap to PI ceiling after optimisation:')
print(f'  SARSA:      {pi_results["survival_rate"] - sarsa_opt_results["survival_rate"]:.1f}pp')
print(f'  Q-Learning: {pi_results["survival_rate"] - ql_opt_results["survival_rate"]:.1f}pp')
print()
print('This remaining gap represents the fundamental cost of not having')
print('the full MDP model — a model-free agent cannot fully close it')
print('without perfect environment knowledge.')
